# 06 - Verification: self-tests, traces, compatibility, overhead, gates

**Purpose:** Run ML-KEM / ML-DSA self-tests, before/after loopback TLS traces and the compatibility matrix with OpenSSL 3.5; chart handshake bytes, compatibility, timing distribution and gate states.

**Inputs:** Git-bundled OpenSSL 3.5.9 (full path), throw-away test keys in a temp dir

**Outputs:** self-test table, bytes chart, compatibility chart, timing box plot, gate heatmap

**Status labels:** VERIFIED, TEST SCENARIO (loopback, test keys). Self-tests are round-trip tests, NOT NIST KATs. Timing is loopback process time, not a network benchmark.

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
from cryptoshift.gates import openssl_lab as lab
print(lab.capabilities())
with lab.lab_dir() as d:
    st = [lab.mlkem_roundtrip_openssl(d), lab.mlkem_roundtrip_cryptography(), lab.mldsa_sign_verify_openssl(d), lab.mldsa_sign_verify_cryptography()]
    tr = lab.traces(d)
    cm = lab.compatibility_matrix(d)
pd.DataFrame(st)[['implementation','algorithm','status','note']]

In [ ]:
tdf = pd.DataFrame([{'trace': k, **{f: v.get(f) for f in ('handshake_ok','protocol','group','peer_signature_type','bytes_read','bytes_written')}} for k, v in tr.items() if isinstance(v, dict) and 'handshake_ok' in v]).set_index('trace')
ax = tdf[['bytes_read','bytes_written']].rename(index={'before_classical':'before: X25519\n(EC cert)','after_hybrid':'after: X25519MLKEM768\n(EC cert)','after_hybrid_mldsa_cert':'after: X25519MLKEM768\n(ML-DSA-65 cert)'}).plot(kind='bar', figsize=(10, 4), color=['#457b9d','#e9c46a'], title='Handshake bytes seen by the client (loopback test scenario)')
for cont in ax.containers: ax.bar_label(cont, fmt='%d', fontsize=8)
ax.set_ylabel('bytes'); ax.tick_params(axis='x', rotation=0); savefig('06_handshake_bytes')
tdf

In [ ]:
cdf = pd.DataFrame(cm)
fig, ax = plt.subplots(figsize=(11, 4.5))
col = {'positive':'#2a9d8f','fallback':'#457b9d','negative':'#e76f51'}
ax.barh(cdf['case'], [1]*len(cdf), color=[col[c] for c in cdf['category']])
for i, r in cdf.iterrows(): ax.text(0.02, i, ('PASS ' if r['pass'] else 'FAIL ') + r['observed'][:70], va='center', fontsize=7.5, color='white')
ax.set_xticks([]); ax.invert_yaxis(); ax.grid(False)
ax.set_title(f"Compatibility matrix: {cdf['pass'].sum()}/{len(cdf)} cases matched expectation (green positive, blue fallback, red must be rejected)")
savefig('06_compatibility')
cdf[['case','category','expected','observed','pass']]

In [ ]:
ev = json.loads(ensure('scratch/derived/member3/verification_evidence.json', 'member3_phase4.py').read_text())
res = ev['overhead']['results']
if not all('samples_ms' in v for v in res.values()):
    subprocess.run([PY, str(ROOT / 'scripts' / 'member3_phase4.py')], cwd=ROOT, check=True)
    ev = json.loads((DERIVED / 'verification_evidence.json').read_text()); res = ev['overhead']['results']
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([res[k]['samples_ms'] for k in res], tick_labels=list(res), showfliers=True)
ax.set_ylabel('ms per s_client process (incl. start-up)'); ax.set_title(ev['overhead']['method'][:90] + '...', fontsize=8)
savefig('06_overhead_boxplot')
pd.DataFrame({k: {m: v[m] for m in ('n_ok','median_ms','p95_ms','min_ms')} for k, v in res.items()})

In [ ]:
from cryptoshift.gates.verification_gates import evaluate_plan
from cryptoshift.planning.planner import build_problem
steps = evaluate_plan(build_problem()['services'], ev)
gates = sorted({g['gate'] for s in steps for g in s['gates']})
val = {'PASS': 2, 'FAIL': 0, 'BLOCKED': 1}
mat = pd.DataFrame([[val.get(next((g['status'] for g in s['gates'] if g['gate']==k), None), float('nan')) for k in gates] for s in steps], index=[f"{s['service_id']} ({s['state']})" for s in steps], columns=gates)
fig, ax = plt.subplots(figsize=(11, 3.8))
from matplotlib.colors import ListedColormap
ax.imshow(mat.values, cmap=ListedColormap(['#e76f51','#e9c46a','#2a9d8f']), vmin=0, vmax=2, aspect='auto')
ax.set_xticks(range(len(gates)), gates, rotation=30, ha='right'); ax.set_yticks(range(len(mat)), mat.index); ax.grid(False)
for (i, j), v in __import__('numpy').ndenumerate(mat.values): ax.text(j, i, {2:'PASS',1:'BLOCKED',0:'FAIL'}.get(v, '-'), ha='center', va='center', fontsize=8, color='white' if v==v else 'black')
ax.set_title('Gate results per migration step (blank = gate not required for that step)')
savefig('06_gate_heatmap')

READY means the generic loopback evidence passed; it does not exercise each service's own stack (KMS wrapping, JWT libraries, partner peers). Thresholds are assumptions (docs/member3/assumptions.md). Process start-up dominates the timings; byte counts are the reliable overhead signal.